In [1]:
import torch
import random
import gc
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import numpy as np

from biojepa_v0_7 import BioJepa, BioJepaConfig
from training_v0_7 import create_model, maybe_compile
from config_v0_7 import DataConfig
from evals.evals import EvalContext, run_encoder_evals, run_composer_evals, run_ac_evals, save_report
from evals.linear_expression_decoder import BenchmarkDecoder, BenchmarkDecoderConfig

In [2]:
SEED = 1337

def get_device():
    device = 'cpu'
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)
        device = 'cuda'
    print(f'using {device}')
    return torch.device(device)

torch.manual_seed(SEED)
random.seed(SEED)
torch.set_float32_matmul_precision('high')

device = get_device()

USE_AMP = torch.cuda.is_available()
USE_COMPILE = torch.cuda.is_available()
USE_COMPILE = False
USE_FUSED = torch.cuda.is_available()

data_root = Path('~/data/v0_7').expanduser()
ref_root = Path('~/data/reference_data').expanduser()

data_cfg = DataConfig(
    data_root=data_root,
    checkpoint_dir=data_root / 'checkpoint',
    ref_dir=ref_root,
    eval_results_dir=data_root / 'eval_results'
)

using cuda


In [3]:
model_cfg = BioJepaConfig(
    num_genes=10000,
    n_layer=6,
    heads=4,
    embed_dim=256,
    mlp_ratio=4.0,
    n_pre_layer=2,
    mask_ratio=0.766,
    gaussian_scale=5.699,
    film_linear_multiple=0.6769,
    sim_coeff=50.18,
    std_coeff=25.44,            # sim_coeff * std_to_sim_ratio (0.5069)
    cov_coeff=0.5158,           # sim_coeff * cov_to_sim_ratio (0.01028)
    pert_latent_dim=128,
    pert_mode_dim=64,
    predictor_embed_dim=128,
    predictor_n_layer=4,
    predictor_heads=4,
)


EVAL_BATCH_SIZE = 64

In [4]:
model = create_model(model_cfg, device)
model = maybe_compile(model, USE_COMPILE)

print(f'Student/Teacher: {sum(p.numel() for p in model.student.parameters()):,}')
print(f'ACpredictor: {sum(p.numel() for p in model.predictor.parameters()):,}')
print(f'PerturbationComposer: {sum(p.numel() for p in model.composer.parameters()):,}')

Student/Teacher: 7,979,650
ACpredictor: 2,565,760
PerturbationComposer: 444,224


In [5]:
USE_COMPILE

False

### Load Model 

In [6]:
checkpoint_path = data_cfg.checkpoint_dir / 'biojepa_evaled_final.pt'
with torch.serialization.safe_globals([BioJepaConfig]):
    checkpoint = torch.load(checkpoint_path, map_location=device)

state_dict = checkpoint['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in state_dict):
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}

keys = model.load_state_dict(state_dict)
keys

<All keys matched successfully>

### Encoder Training Evals

In [ ]:
eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
pt_eval_results = run_encoder_evals(eval_ctx)



In [ ]:
save_report(pt_eval_results, data_cfg.eval_results_dir / 'encoder_eval_report.json')
pt_eval_results

In [ ]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()

### Alignment Training Eval

In [ ]:
align_eval_ctx = EvalContext.from_trained_model(model, decoder=None, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED
})
align_eval_results = run_composer_evals(align_eval_ctx)

save_report(align_eval_results, data_cfg.eval_results_dir / 'composer_eval_report.json')
align_eval_results

In [ ]:
del align_eval_ctx
gc.collect()
torch.cuda.empty_cache()

### ACPredictor Eval

In [7]:
decoder_path = data_cfg.checkpoint_dir / 'biojepa_evaled_decoder_final.pt'
decoder_ckpt = torch.load(decoder_path, map_location=device)

decoder_sd = decoder_ckpt['model']
if not USE_COMPILE and any('_orig_mod.' in k for k in decoder_sd):
    decoder_sd = {k.replace('_orig_mod.', ''): v for k, v in decoder_sd.items()}

decoder = BenchmarkDecoder(BenchmarkDecoderConfig(embed_dim=model_cfg.embed_dim)).to(device)
keys = decoder.load_state_dict(decoder_sd)
keys

<All keys matched successfully>

In [8]:
eval_ctx = EvalContext.from_trained_model(model, decoder=decoder, data_root=data_cfg.data_root, ref_dir=data_cfg.ref_dir, config={
    'num_genes': model_cfg.num_genes, 'embed_dim': model_cfg.embed_dim,
    'n_layer': model_cfg.n_layer, 'heads': model_cfg.heads, 'batch_size': EVAL_BATCH_SIZE, 'seed': SEED,
})
full_eval_results = run_ac_evals(eval_ctx)

save_report(full_eval_results, data_cfg.eval_results_dir / 'ac_eval_report.json')
full_eval_results

Using cuda
found 135 shards for split test


Running test inference:   0%|                                                              | 0/5400 [00:00<?, ?it/s]

Loaded DNA seq bank: torch.Size([11643, 1536])
Loaded chemical seq bank: torch.Size([188, 1536])
Loaded target bank: torch.Size([9975, 320])


Running test inference:   1%|▍                                                  | 40/5400 [00:20<4:25:29,  2.97s/it]/home/ubuntu/code/biojepa/evals/evals.py:643: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  corr, _ = pearsonr(p_top, t_top)
Running test inference: 100%|███████████████████████████████████████████████████| 5400/5400 [44:36<00:00,  2.02it/s]


Aggregated 1416 single-pert, 13 multi-pert perturbations, 345600 samples, 135 shards
  adamson: 11 perturbations, 4288 samples
  k562e_raw: 286 perturbations, 46506 samples
  k562gw: 1053 perturbations, 189095 samples
  norman: 10 perturbations, 9472 samples
  rep1e: 287 perturbations, 22838 samples
  sciplex: 54 perturbations, 73401 samples
Cached test inference to /home/ubuntu/data/v0_7/test_inference_cache (135 shards)
expression_prediction: Pearson=0.9873, R2=0.9680, Centroid_acc=0.1492
gene_level_analysis: Dir_acc=0.9821, Top50_acc=0.7185


perturbation_retrieval (dna): 100%|█████████████████████████████████████████████| 200/200 [2:18:50<00:00, 41.65s/it]


perturbation_retrieval (dna): MRR=0.0007


perturbation_retrieval (chemical): 100%|████████████████████████████████████████████| 54/54 [00:36<00:00,  1.49it/s]


perturbation_retrieval (chemical): MRR=0.0579
Loaded dataset gene masks: ['k562e_raw', 'rep1e', 'k562gw', 'adamson', 'norman', 'sciplex']
uncertainty_calibration: ECE=0.3232, Monotonicity=66.67%
Loading KEGG_2026...
  352 pathways loaded
Loading Reactome_Pathways_2024...
  2100 pathways loaded
moa_matching expression: Within=0.3240, Between=0.3084, Gap=0.0156, Ratio=1.0505x
moa_matching latent: Within=0.6982, Between=0.6930, Gap=0.0052, Ratio=1.0074x
Loaded Norman combo mapping: 132 combos
Loaded Norman single-gene deltas: 105 genes
Loaded Norman GI subtypes: 88 combos
Loaded dataset splits: ['k562e_raw', 'rep1e', 'k562gw', 'adamson', 'norman', 'sciplex']
combination_perturbation: 13 combo perts, 4283 samples, 13 additive baseline, 5 GI-labeled, 13 generalization-classified
dose_response: monotonicity=50.00%, real_mono=48.77%, spearman=-0.0212, curve_sim=0.5237161206803241
Saved report to /home/ubuntu/data/v0_7/eval_results/ac_eval_report.json


{'expression_prediction': {'config': {'test_perturbations': 1416,
   'genes': 10000,
   'test_samples': 345600},
  'sample_level': {'mse': 0.19492395973606982,
   'pearson_r_top20': 0.8654291389368788},
  'perturbation_level': {'r2_all_genes': {'mean': 0.9679888117700647,
    'median': 0.9806722402572632},
   'r2_top50_degs': {'mean': 0.8077260832917892, 'median': 0.8957712948322296},
   'mse': {'mean': 0.0072744437493383884, 'median': 0.0041872914880514145},
   'pearson_all_genes': {'mean': 0.987335889352917,
    'median': 0.9932138323783875},
   'pearson_delta_all_genes': {'mean': 0.3324136134055522,
    'median': 0.3330240994691849},
   'pearson_top50_degs': {'mean': 0.5993347961776548,
    'median': 0.6364570260047913}},
  'centroid_accuracy': {'accuracy': 0.1492087415222306, 'n_groups': 1327},
  'vs_baseline': {'beat_rate': 0.2860169491525424, 'n_evaluated': 1416},
  'severity': {'pearson_r': 0.797481894493103,
   'spearman_r': 0.7807300480239583},
  'error_by_magnitude': {'0-0.25

In [ ]:
del eval_ctx
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import hashlib
import inspect

import torch

import biojepa_v0_7
import dataloader_v0_7
import evals.evals as evals_module
import evals.linear_expression_decoder as decoder_module

from dataloader_v0_7 import EvalLoader
from evals.evals import get_seq_embeddings, get_target_embeddings


def state_hash(module):
    canonical = {
        key.replace("_orig_mod.", ""): value
        for key, value in module.state_dict().items()
    }
    digest = hashlib.sha256()
    for key, value in sorted(canonical.items()):
        tensor = value.detach().cpu().contiguous()
        digest.update(key.encode())
        digest.update(tensor.view(torch.uint8).numpy().tobytes())
    return digest.hexdigest()[:16]


def tensor_signature(tensor):
    sample = tensor.detach().reshape(-1)[:4096].float().cpu()
    digest = hashlib.sha256(sample.numpy().tobytes()).hexdigest()[:16]
    return {
        "shape": tuple(tensor.shape),
        "mean": float(sample.mean()),
        "std": float(sample.std(unbiased=False)),
        "min": float(sample.min()),
        "max": float(sample.max()),
        "sha": digest,
    }


def source_hash(obj):
    return hashlib.sha256(inspect.getsource(obj).encode()).hexdigest()[:16]


print("FILES")
print("model:", biojepa_v0_7.__file__)
print("loader:", dataloader_v0_7.__file__)
print("evals:", evals_module.__file__)
print("decoder:", decoder_module.__file__)
print("torch:", torch.__version__)

print("\nSTATE")
print("model:", state_hash(model))
print("decoder:", state_hash(decoder))

print("\nFLAGS BEFORE FORCING EVAL")
print("model:", model.training)
print("teacher:", model.teacher.training)
print("composer:", model.composer.training)
print("composer dropout:", model.composer.latent_dropout.training)
print("predictor:", model.predictor.training)
print("decoder:", decoder.training)

print("\nSOURCE")
print("teacher:", source_hash(biojepa_v0_7.CellStateEncoder.forward))
print("composer:", source_hash(biojepa_v0_7.ActionComposer.forward))
print("predictor:", source_hash(biojepa_v0_7.ACPredictor.forward))
print("decoder:", source_hash(decoder_module.BenchmarkDecoder.forward))
print("inference:", source_hash(evals_module.EvalContext._run_test_inference))

model.eval()
decoder.eval()

print("\nBANKS")
print("dna:", tensor_signature(eval_ctx.seq_banks["dna"]))
print("target:", tensor_signature(eval_ctx.target_bank))

loader = EvalLoader(
    batch_size=64,
    split="test",
    data_dir=data_cfg.data_root / "predictor_t",
    device=device,
    seed=1337,
)
batch = loader.next_batch()

print("\nINPUT")
print("control:", tensor_signature(batch.control))
print("seq_idx:", tensor_signature(batch.seq_idx))
print("target_idx:", tensor_signature(batch.target_idx))
print("dose:", tensor_signature(batch.dose))

with torch.no_grad():
    pert_mask = (
        torch.arange(batch.seq_idx.shape[1], device=device).unsqueeze(0)
        < batch.n_perts.unsqueeze(1)
    )
    seq_emb = get_seq_embeddings(
        batch.seq_idx, batch.modality, eval_ctx.seq_banks
    )
    target_emb = get_target_embeddings(
        batch.target_idx, eval_ctx.target_bank
    )
    unknown_mask = ~batch.gene_mask

    z_context = model.teacher(
        batch.control,
        batch.control_total,
        mask_idx=None,
        unknown_mask=unknown_mask,
    )
    action = model.composer(
        seq_emb,
        target_emb,
        batch.modality,
        batch.mode,
        batch.has_seq,
        batch.has_target,
        pert_mask,
        dose=batch.dose,
    )
    target_indices = torch.arange(
        batch.control.shape[1], device=device
    ).expand(batch.control.shape[0], -1)
    z_pred, _ = model.predictor(z_context, action, target_indices)
    pred_delta = decoder(z_pred) - decoder(z_context)

print("\nFORWARD")
print("seq_emb:", tensor_signature(seq_emb))
print("target_emb:", tensor_signature(target_emb))
print("context:", tensor_signature(z_context))
print("action:", tensor_signature(action))
print("prediction:", tensor_signature(z_pred))
print("pred_delta:", tensor_signature(pred_delta))


In [ ]:
teacher_eager = model.teacher._orig_mod
teacher_eager.eval()

with torch.no_grad():
  eager_context = teacher_eager(
      batch.control,
      batch.control_total,
      mask_idx=None,
      unknown_mask=unknown_mask,
  )

print("compiled:", tensor_signature(z_context))
print("eager:", tensor_signature(eager_context))
print(
  "max difference:",
  float((z_context.float() - eager_context.float()).abs().max()),
)